# GDN rank versus context length

The 4B model fits a 24 GiB GPU. Each document uses one capture pass; all GDN layers share the U recurrence kernels; only aggregates leave the GPU. Ranks are normalized by `min(matrix.shape)`.

In [ ]:
import math
import operator
from pathlib import Path
from typing import Optional

import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch._dynamo import lookup_backend

from fla.ops.gated_delta_rule import chunk_gated_delta_rule as real_gdn


# ============================================================
# Config
# ============================================================

MODEL_ID = "Qwen/Qwen3.5-4B"

N_SAMPLES = 1000
LENGTHS = (8, 32, 128, 512, 4096)
SEED = 0

# Short sequences need much larger batches to utilize the GPU.
# Adjust upward if you still have lots of free VRAM.
BATCH_SIZES = {
    8: 64,
    32: 64,
    128: 32,
    512: 16,
    4096: 8,
}

MAX_LENGTH = max(LENGTHS)
TOKEN_CACHE = Path(
    f"fineweb_tokens_{N_SAMPLES}_{MAX_LENGTH}_{SEED}.pt"
)

DEVICE = "cuda"

torch.set_grad_enabled(False)


# ============================================================
# Load HF model directly
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map=None,
).cuda().eval()

layer_ids = [
    i
    for i, layer in enumerate(model.model.layers)
    if hasattr(layer, "linear_attn")
]

print(f"GDN layers ({len(layer_ids)}): {layer_ids}")


# ============================================================
# Opaque FLA GDN custom op
#
# Critical point:
# Dynamo sees this as one stable graph node with:
#
#        Q,K,V,g,beta -> GDN -> output,S
#
# It does NOT trace inside FLA.
# ============================================================

@torch.library.custom_op(
    "mechinterp::gdn",
    mutates_args=(),
)
def opaque_gdn(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    g: torch.Tensor,
    beta: torch.Tensor,
    initial_state: Optional[torch.Tensor] = None,
    output_final_state: bool = False,
    use_qk_l2norm_in_kernel: bool = True,
    cu_seqlens: Optional[torch.Tensor] = None,
) -> tuple[torch.Tensor, torch.Tensor]:

    # Always ask FLA for S so it exists as a graph output.
    out, state = real_gdn(
        q,
        k,
        v,
        g=g,
        beta=beta,
        initial_state=initial_state,
        output_final_state=True,
        use_qk_l2norm_in_kernel=use_qk_l2norm_in_kernel,
        cu_seqlens=cu_seqlens,
    )

    return out, state


@opaque_gdn.register_fake
def _fake_gdn(
    q,
    k,
    v,
    g,
    beta,
    initial_state=None,
    output_final_state=False,
    use_qk_l2norm_in_kernel=True,
    cu_seqlens=None,
):
    B, T, H, DK = k.shape
    DV = v.shape[-1]

    out = torch.empty_like(v)

    state = torch.empty(
        B,
        H,
        DK,
        DV,
        device=v.device,
        dtype=v.dtype,
    )

    return out, state


# ============================================================
# FX instrumentation backend
# ============================================================

PROBES = {}

inductor = lookup_backend("inductor")


def make_backend(layer_idx):

    def backend(gm, example_inputs):
        nodes = list(gm.graph.nodes)

        gdn_node = next(
            (
                n
                for n in nodes
                if n.op == "call_function"
                and "mechinterp.gdn" in str(n.target)
            ),
            None,
        )

        # Other graph fragments, e.g. causal_conv1d.
        if gdn_node is None:
            return inductor(gm, example_inputs)

        # opaque_gdn(q, k, v, ...)
        key = gdn_node.args[1]

        output_node = next(
            n for n in gm.graph.nodes
            if n.op == "output"
        )

        original_output = output_node.args[0]

        if not isinstance(original_output, tuple):
            raise RuntimeError(
                f"Unexpected FX output structure: "
                f"{type(original_output)}"
            )

        n_original = len(original_output)

        with gm.graph.inserting_before(output_node):

            # ------------------------------------------------
            # Final recurrent state
            #
            # S: [B,H,DK,DV]
            # ------------------------------------------------

            state = gm.graph.call_function(
                operator.getitem,
                args=(gdn_node, 1),
            )

            # ------------------------------------------------
            # K^T K
            #
            # K:
            #   [B,T,H,D]
            # -> [B,H,T,D]
            # ------------------------------------------------

            k_bhtd = gm.graph.call_method(
                "permute",
                args=(key, 0, 2, 1, 3),
            )

            kt = gm.graph.call_method(
                "transpose",
                args=(k_bhtd, -1, -2),
            )

            kgram = gm.graph.call_function(
                torch.matmul,
                args=(kt, k_bhtd),
            )

            # ------------------------------------------------
            # S S^T
            #
            # S:
            #   [B,H,DK,DV]
            # -> [B,H,DK,DK]
            # ------------------------------------------------

            st = gm.graph.call_method(
                "transpose",
                args=(state, -1, -2),
            )

            sgram = gm.graph.call_function(
                torch.matmul,
                args=(state, st),
            )

            # Eigendecomposition is more reliable in fp32.
            # The large K/S tensors remain fp16.
            kgram = gm.graph.call_method(
                "float",
                args=(kgram,),
            )

            sgram = gm.graph.call_method(
                "float",
                args=(sgram,),
            )

        # Temporarily expose probes through graph outputs.
        output_node.args = (
            (*original_output, kgram, sgram),
        )

        gm.graph.lint()
        gm.recompile()

        # Actually compile the rewritten graph.
        compiled = inductor(
            gm,
            example_inputs,
        )

        def run(*args):
            out = compiled(*args)

            # Keep tensors on GPU.
            # No .cpu() or .item() here.
            PROBES[layer_idx] = (
                out[-2].detach(),
                out[-1].detach(),
            )

            return tuple(out[:n_original])

        return run

    return backend


# ============================================================
# Patch kernel boundary + compile each GDN independently
# ============================================================
import torch._dynamo as dynamo
dynamo.config.recompile_limit = 64

for layer_idx in layer_ids:
    gdn = model.model.layers[layer_idx].linear_attn

    gdn.chunk_gated_delta_rule = opaque_gdn

    model.model.layers[layer_idx].linear_attn = torch.compile(
        gdn,
        backend=make_backend(layer_idx),
        fullgraph=False,
        dynamic=True,
    )


# ============================================================
# Token cache
# ============================================================

if TOKEN_CACHE.exists():
    print(f"Loading token cache: {TOKEN_CACHE}")
    tokens = torch.load(
        TOKEN_CACHE,
        map_location="cpu",
    )

else:
    print("Building token cache...")

    ds = (
        load_dataset(
            "HuggingFaceFW/fineweb-edu",
            "sample-10BT",
            split="train",
            streaming=True,
        )
        .shuffle(
            seed=SEED,
            buffer_size=10_000,
        )
    )

    token_rows = []

    pbar = tqdm(
        total=N_SAMPLES,
        desc="Tokenizing",
        unit="doc",
    )

    for row in ds:
        ids = tokenizer(
            row["text"],
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH,
            add_special_tokens=True,
        ).input_ids[0]

        if ids.numel() < MAX_LENGTH:
            continue

        token_rows.append(
            ids[:MAX_LENGTH].contiguous()
        )

        pbar.update(1)

        if len(token_rows) >= N_SAMPLES:
            break

    pbar.close()

    tokens = torch.stack(token_rows)

    torch.save(
        tokens,
        TOKEN_CACHE,
    )

print(
    f"tokens: {tuple(tokens.shape)} "
    f"{tokens.dtype}"
)


# ============================================================
# Statistics from Gram matrix
#
# If G = X X^T or X^T X, eigenvalues(G) = singular_values(X)^2.
# ============================================================

def gram_stats(gram):
    """
    gram: [B,H,D,D]

    returns [B,H] tensors.
    """

    # Numerical noise can produce tiny negative eigenvalues.
    eig = torch.linalg.eigvalsh(gram)
    eig = eig.clamp_min_(0)

    total = eig.sum(dim=-1)
    largest = eig[..., -1].clamp_min(1e-30)

    # ||X||_F
    fro_norm = total.sqrt()

    # ||X||_2
    spectral_norm = largest.sqrt()

    # ||X||_F^2 / ||X||_2^2
    stable_rank = total / largest

    # Entropy effective rank.
    p = eig / total.unsqueeze(-1).clamp_min(1e-30)

    entropy = -(
        p
        * p.clamp_min(1e-30).log()
    ).sum(dim=-1)

    effective_rank = entropy.exp()

    # Components needed to explain 90% of squared singular value mass.
    desc = eig.flip(-1)
    cumulative = desc.cumsum(dim=-1)

    threshold = (
        0.9 * total.unsqueeze(-1)
    )

    r90 = (
        cumulative < threshold
    ).sum(dim=-1) + 1

    return torch.stack(
        [
            fro_norm,
            spectral_norm,
            stable_rank,
            effective_rank,
            r90.float(),
        ],
        dim=-1,
    )


STAT_NAMES = [
    "fro_norm",
    "spectral_norm",
    "stable_rank",
    "effective_rank",
    "r90",
]


# ============================================================
# Results storage
#
# Store CPU tensors per batch, not individual Python scalars.
# ============================================================

results = []


# ============================================================
# Warmup
#
# Compile once before starting timing/progress.
# 128 is enough to instantiate the relevant dynamic graphs.
# ============================================================

print("Compiling / warming up...")

warmup_ids = tokens[:1, :128].to(
    DEVICE,
    non_blocking=True,
)

PROBES.clear()

with torch.inference_mode():
    _ = model.model(
        input_ids=warmup_ids,
        use_cache=False,
        return_dict=True,
    )

torch.cuda.synchronize()

print("Warmup complete.")


# ============================================================
# Main run
# ============================================================

total_steps = sum(
    math.ceil(N_SAMPLES / BATCH_SIZES[length])
    for length in LENGTHS
)

pbar = tqdm(
    total=total_steps,
    desc="GDN statistics",
    unit="batch",
    dynamic_ncols=True,
)


with torch.inference_mode():

    for length in LENGTHS:

        batch_size = BATCH_SIZES[length]

        for start in range(
            0,
            N_SAMPLES,
            batch_size,
        ):
            stop = min(
                start + batch_size,
                N_SAMPLES,
            )

            ids = tokens[
                start:stop,
                :length,
            ].to(
                DEVICE,
                non_blocking=True,
            )

            PROBES.clear()

            # Backbone only:
            # avoids allocating enormous vocabulary logits.
            _ = model.model(
                input_ids=ids,
                use_cache=False,
                return_dict=True,
            )

            missing = (
                set(layer_ids)
                - set(PROBES.keys())
            )

            if missing:
                raise RuntimeError(
                    f"Missing probes for layers: "
                    f"{sorted(missing)}"
                )

            # --------------------------------------------
            # Do all expensive linear algebra on GPU.
            # Transfer only [B,H,5] statistics afterward.
            # --------------------------------------------

            for layer_idx in layer_ids:

                kgram, sgram = PROBES[layer_idx]

                kstats = gram_stats(kgram)
                sstats = gram_stats(sgram)

                # Async-ish host transfer at a coarse granularity.
                results.append(
                    (
                        start,
                        stop,
                        length,
                        layer_idx,
                        kstats.cpu(),
                        sstats.cpu(),
                    )
                )

            del ids
            PROBES.clear()

            pbar.update(1)

            pbar.set_postfix(
                length=length,
                batch=stop - start,
                vram=f"{torch.cuda.memory_allocated()/2**30:.1f}G",
            )


pbar.close()

torch.cuda.synchronize()


# ============================================================
# Convert tensors -> DataFrame AFTER GPU computation finishes
# ============================================================

print("Building DataFrame...")

rows = []

for (
    start,
    stop,
    length,
    layer_idx,
    kstats,
    sstats,
) in tqdm(
    results,
    desc="Formatting",
    unit="batch",
):

    for matrix_name, stats in (
        ("K", kstats),
        ("S", sstats),
    ):
        B, H, _ = stats.shape

        # This loop is now CPU-only and therefore doesn't
        # synchronize CUDA thousands of times.
        for b in range(B):
            sample_idx = start + b

            for head in range(H):
                values = stats[b, head]

                rows.append({
                    "sample": sample_idx,
                    "length": length,
                    "layer": layer_idx,
                    "head": head,
                    "matrix": matrix_name,

                    "fro_norm": float(values[0]),
                    "spectral_norm": float(values[1]),
                    "stable_rank": float(values[2]),
                    "effective_rank": float(values[3]),
                    "r90": int(values[4]),
                })


df = pd.DataFrame(rows)

print(df.head())
print(f"\nrows: {len(df):,}")

print(
    "\nPeak CUDA memory:",
    f"{torch.cuda.max_memory_allocated()/2**30:.2f} GiB",
)


# Optional:
# df.to_parquet("gdn_k_s_stats.parquet", index=False)

In [ ]:
import plotly.graph_objects as go

MATRICES = ("K", "S")

LABELS = {
    "effective_rank": "Effective rank / dimension",
    "stable_rank": "Stable rank / dimension",
    "r90": "r90 / dimension",
}

# Maximum possible rank for normalization.
#
# K is [T, D], D=128 -> rank <= min(T, 128)
# S is [128, 128]   -> rank <= 128
plot_df = df.copy()

for metric in LABELS:
    denom = plot_df.apply(
        lambda row:
            min(row["length"], 128)
            if row["matrix"] == "K"
            else 128,
        axis=1,
    )

    plot_df[f"{metric}_normalized"] = (
        plot_df[metric] / denom
    )


for metric, label in LABELS.items():

    value_col = f"{metric}_normalized"

    figure = go.Figure()
    ids = sorted(plot_df.layer.unique())

    for layer_index, layer in enumerate(ids):

        for matrix_index, matrix in enumerate(MATRICES):

            part = plot_df[
                (plot_df.layer == layer)
                & (plot_df.matrix == matrix)
            ]

            # Mean over documents.
            mean = (
                part
                .groupby(["head", "length"])[value_col]
                .mean()
                .reset_index()
            )

            pivot = mean.pivot(
                index="head",
                columns="length",
                values=value_col,
            )

            # Preserve configured length order.
            pivot = pivot.reindex(
                columns=[
                    x for x in LENGTHS
                    if x in pivot.columns
                ]
            )

            figure.add_trace(
                go.Heatmap(
                    x=pivot.columns,
                    y=pivot.index,
                    z=pivot.to_numpy(),
                    colorscale="Viridis",
                    zmin=0,
                    zmax=1,
                    colorbar={"title": label},
                    visible=(
                        layer_index == 0
                        and matrix_index == 0
                    ),
                    hovertemplate=(
                        f"layer {layer}<br>"
                        "head %{y}<br>"
                        "tokens %{x}<br>"
                        f"{label}: %{{z:.3f}}"
                        f"<extra>{matrix}</extra>"
                    ),
                )
            )

    buttons = []

    for layer_index, layer in enumerate(ids):
        for matrix_index, matrix in enumerate(MATRICES):

            visible = [False] * len(figure.data)

            visible[
                layer_index * len(MATRICES)
                + matrix_index
            ] = True

            buttons.append({
                "label": f"layer {layer} · {matrix}",
                "method": "update",
                "args": [
                    {"visible": visible},
                    {
                        "title":
                            f"{label} — layer {layer}, {matrix}"
                    },
                ],
            })

    figure.update_layout(
        title=(
            f"{label} — "
            f"layer {ids[0]}, {MATRICES[0]}"
        ),
        xaxis_title="Text length (tokens)",
        yaxis_title="Value head",
        updatemenus=[{
            "buttons": buttons,
            "x": 0,
            "y": 1.16,
            "xanchor": "left",
        }],
        height=620,
    )

    figure.show()